# PySpark

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/spark.ipynb)

Five parts: what PySpark is, the execution model, shaping data, production patterns, and how to
answer in an interview. Every code cell runs on a local Spark session against a small sample
dataset, so predict each output before you run the cell.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Part 1 - Orientation: what PySpark is and where it fits

No code in this part: it defines the words every later section uses - driver and executor,
DataFrame and RDD, lazy plan.

## What is PySpark and where does it fit in a data platform?

PySpark is the Python API for Apache Spark. It lets you write distributed data processing jobs in Python while Spark executes the work across a cluster.

In a modern lakehouse, PySpark is often used for:
+ Reading raw files from S3, ADLS, GCS, Kafka, or relational sources
+ Cleaning and standardizing semi-structured data
+ Joining large fact data with dimensions
+ Writing Delta, Iceberg, or Parquet tables for analytics
+ Running batch pipelines, streaming pipelines, and feature engineering jobs

Typical production flow:
```text
raw source files -> PySpark bronze job -> PySpark silver job -> Unity Catalog gold tables -> BI / ML / reverse ETL
```

Real-world example: an orders pipeline lands JSON in S3 every 15 minutes. A PySpark job reads the raw JSON, enforces a schema, deduplicates by `order_id`, joins customer data, and writes a governed table such as `main.gold.fact_order`.

## How do DataFrames differ from RDDs in PySpark?

DataFrames are structured, schema-aware distributed datasets. Spark can optimize DataFrame operations using Catalyst and Tungsten. RDDs are lower-level distributed collections where Spark sees less structure and has fewer optimization opportunities.

How it works under the hood. Catalyst sees DataFrame expressions and rewrites them with rules such as predicate pushdown, column pruning and constant folding; Tungsten then compiles the physical operators into one generated Java method over a compact binary row format. An RDD holds opaque Python or JVM objects, so the optimizer cannot see inside a lambda, and `df.rdd` pays a per-row conversion between JVM rows and Python objects.

Worked example. A `filter` on a DataFrame compiles into a single generated method over binary rows. The same predicate written as `df.rdd.filter(lambda r: ...)` ships Python objects across the Py4J boundary for every row and can be 5-20x slower on a wide table.

Common mistake: `df.rdd.map(...).toDF()` to express logic that `pyspark.sql.functions` already covers. You keep the syntax you know but lose predicate pushdown and column pruning.

## When you would NOT use PySpark

Spark is a heavy default, not a habit. Reach for something simpler when:

+ the data fits in one machine's memory (a few GB) - pandas, DuckDB or plain SQL is faster to write, run and iterate on;
+ the work is pure relational SQL over warehouse tables - the warehouse engine (or dbt) avoids shipping and maintaining Python for what SQL already does;
+ latency-sensitive serving and row lookups are the job - pipelines are batch animals; point lookups belong in a database;
+ the source system can do the transform for you (a streaming engine, a view), because every extra engine multiplies the operational surface.

The honest two-line interview answer: "PySpark earns its cost once data volume, streaming,
or heavy custom Python logic appears; before that, simpler engines win."

## Part 2 - First session and the execution model

Everything above was words; here the machine appears. One idea to carry through this
part: **nothing runs until it must**. Create the session, seed the small dataset the
lessons below share, read with an explicit schema, transform lazily, and only then
pull the trigger with an action.

## How do you create a SparkSession in PySpark?

`SparkSession` is the main entry point for DataFrame, SQL, catalog, and streaming operations. In Databricks notebooks, `spark` is usually already available. In jobs, tests, or local development, you create it explicitly.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

On Databricks with Unity Catalog the session already exists and tables have three-part names:

```python
orders = spark.table("main.silver.orders")
```

This notebook uses a local session catalog instead, so table names below have one or two parts.

How it works under the hood. `SparkSession.builder` reads static configs, then `getOrCreate()` returns the existing session if one is already running in this JVM. Configs passed to the builder are fixed for the life of the session; `spark.conf.set(...)` can change runtime SQL settings later. Every shuffle uses `spark.sql.shuffle.partitions` as its reduce fan-out.

Worked example, the defaults you should know:

```text
spark.sql.shuffle.partitions          200
spark.sql.autoBroadcastJoinThreshold  10485760 bytes (10 MB)
spark.sql.adaptive.enabled            true (Spark 3.2+)
spark.sql.files.maxPartitionBytes     134217728 bytes (128 MB)
```

Common mistake: calling `getOrCreate()` again with different configs and expecting them to apply. It silently returns the first session, so stop the old one or use `spark.conf.set` for runtime settings.

How it works under the hood. A PySpark program has one driver process that owns the `SparkSession`, and one or more executor processes that own the data. The driver turns your DataFrame calls into a logical plan, Catalyst rewrites it, and the DAG scheduler cuts it into stages at every shuffle. Executors run tasks on partitions and report metrics back; `master("local[*]")` packs driver and one executor into a single JVM, which is why it is a development tool and not a deployment topology.

Worked example. A nightly job reads 2 TB of Parquet in 200 files. The driver holds the plan and task metadata (a few MB), while executors hold shuffle files, cached blocks and broadcast tables. With a 2 GB driver and 16 GB executors, the driver limit is the plan plus anything you `collect()`; the executor limit is shuffle, cache and broadcast.

Common mistake: assuming PySpark runs Python on every worker. Python only runs for UDFs and driver-side code; the DataFrame engine runs in the JVM, which is exactly why built-ins are fast and Python UDFs are not.

## Sample data: the tables every lesson below runs on

This section seeds a small four-table slice of an orders platform, built with
`spark.createDataFrame` and an explicit schema per table. Three quirks the lessons lean on:
customer 4 has no `region`, customers 7 and 8 have never ordered, and one event is an exact
retry of the same row.

| table | grain | notable columns |
| --- | --- | --- |
| `customers` | one row per customer | `region`, `signup_date` (`DATE`) |
| `orders` | one row per order | `status`, `order_ts` (`TIMESTAMP`), `amount` (decimal) |
| `order_items` | one row per order line | `qty`, `unit_price` |
| `events` | one row per user event | `user_id`, `event`, `ts` |

### `customers` (8 rows)

Customer 4's `region` never arrived from the CRM - the null-handling cell below fills it.

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import types as T

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), False),
])

customers = spark.createDataFrame(
    [
        (1, "Acme Corp",      "North", date(2024, 1, 3)),
        (2, "Bright Retail",  "North", date(2024, 2, 11)),
        (3, "Cyan Traders",   "South", date(2024, 3, 15)),
        (4, "Delta Grocers",  None,    date(2024, 5, 2)),
        (5, "Echo Supplies",  "West",  date(2024, 6, 20)),
        (6, "Foxtrot Foods",  "West",  date(2024, 8, 1)),
        (7, "Gamma Sports",   "East",  date(2024, 9, 14)),
        (8, "Hotel Homeware", "East",  date(2024, 11, 5)),
    ],
    schema=customers_schema,
)

customers.orderBy("customer_id").show()

### `orders` (12 rows)

`order_ts` is a `TIMESTAMP` and `amount` a decimal, and every order's lines in the next table add
up to its amount exactly. Orders 2 and 3 belong to one customer on the same day, and order 12
lands just after midnight UTC.

In [ ]:
from pyspark.sql import types as T

orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders = spark.createDataFrame(
    [
        (1,  1, "paid",     datetime(2026, 1,  5,  9, 15), D("120.00")),
        (2,  1, "shipped",  datetime(2026, 1,  5, 10,  5),  D("84.50")),
        (3,  1, "returned", datetime(2026, 1,  5, 11, 40),  D("30.00")),
        (4,  2, "paid",     datetime(2026, 1,  6, 15, 20), D("260.00")),
        (5,  2, "placed",   datetime(2026, 1,  7,  8,  5),  D("55.25")),
        (6,  3, "shipped",  datetime(2026, 1,  7, 17, 45), D("149.00")),
        (7,  3, "paid",     datetime(2026, 1,  9, 12, 10), D("310.00")),
        (8,  4, "returned", datetime(2026, 1, 10, 19, 30),  D("89.90")),
        (9,  4, "shipped",  datetime(2026, 1, 12,  8, 55), D("1180.00")),
        (10, 5, "paid",     datetime(2026, 1, 12, 13,  0),   D("74.00")),
        (11, 6, "placed",   datetime(2026, 1, 13, 10, 25),  D("210.00")),
        (12, 6, "shipped",  datetime(2026, 1, 14,  0, 30),   D("96.40")),
    ],
    schema=orders_schema,
)

orders.orderBy("order_id").show()
orders.printSchema()

### `order_items` (17 rows)

SKUs repeat across orders, so `countDistinct("order_id")` has real work to do, and order 9 is a
bulk reorder of twenty units of `SKU-4011`.

In [ ]:
from pyspark.sql import types as T

order_items_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("sku", T.StringType(), False),
    T.StructField("qty", T.IntegerType(), False),
    T.StructField("unit_price", T.DecimalType(8, 2), False),
])

order_items = spark.createDataFrame(
    [
        (1,  "SKU-4001",  2,  D("12.50")),
        (1,  "SKU-4007",  1,  D("95.00")),
        (2,  "SKU-4002",  1,  D("34.50")),
        (2,  "SKU-4009",  2,  D("25.00")),
        (3,  "SKU-4002",  1,  D("30.00")),
        (4,  "SKU-4003",  1, D("149.00")),
        (4,  "SKU-4004",  2,  D("55.50")),
        (5,  "SKU-4005",  1,  D("55.25")),
        (6,  "SKU-4003",  1, D("149.00")),
        (7,  "SKU-4006",  1, D("189.00")),
        (7,  "SKU-4008",  1, D("121.00")),
        (8,  "SKU-4010",  1,  D("89.90")),
        (9,  "SKU-4011", 20,  D("59.00")),
        (10, "SKU-4012",  2,  D("37.00")),
        (11, "SKU-4013",  1, D("210.00")),
        (12, "SKU-4014",  1,  D("41.40")),
        (12, "SKU-4005",  1,  D("55.00")),
    ],
    schema=order_items_schema,
)

order_items.orderBy("order_id", "sku").show(20)

### `events` (13 rows, one is a retry)

One of user 5's `view` events was committed twice by a retried SDK call - the exact shape the
dedupe cell below collapses.

In [ ]:
from pyspark.sql import types as T

events_schema = T.StructType([
    T.StructField("user_id", T.IntegerType(), False),
    T.StructField("event", T.StringType(), False),
    T.StructField("ts", T.TimestampType(), False),
])

events = spark.createDataFrame(
    [
        (1, "view",        datetime(2026, 1,  6,  9,  0)),
        (1, "add_to_cart", datetime(2026, 1,  6,  9,  2)),
        (1, "purchase",    datetime(2026, 1,  6,  9, 10)),
        (2, "view",        datetime(2026, 1,  7, 10,  0)),
        (2, "purchase",    datetime(2026, 1,  7, 10, 40)),
        (3, "view",        datetime(2026, 1,  6, 11, 15)),
        (3, "view",        datetime(2026, 1,  8, 11, 20)),
        (4, "add_to_cart", datetime(2026, 1,  9, 20,  5)),
        (4, "purchase",    datetime(2026, 1, 10,  8, 30)),
        (5, "view",        datetime(2026, 1, 11, 14, 45)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
        (5, "purchase",    datetime(2026, 1, 12, 12, 30)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
    ],
    schema=events_schema,
)

events.orderBy("user_id", "ts").show()

### Row-count sanity check

Four exact counts plus the two invariants the lessons depend on: every order's lines reconcile
with its amount, and the retried event collapses to one row.

In [ ]:
from pyspark.sql import functions as F

frames = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "events": events,
}
expected_rows = {"customers": 8, "orders": 12, "order_items": 17, "events": 13}

for table, frame in frames.items():
    actual = frame.count()
    assert actual == expected_rows[table], (table, actual, expected_rows[table])
    print(f"{table:<12} {actual:>2} rows")

per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines"))
    .join(orders.select("order_id", "amount"), "order_id")
)
assert per_order.where(F.col("lines") != F.col("amount")).count() == 0
print("line totals reconcile with the orders ledger:", float(orders.agg(F.sum("amount")).first()[0]))

assert events.dropDuplicates().count() == events.count() - 1
print("and the retried event collapses to 12 distinct rows - seed verified")

### DataFrames versus RDDs, in code

Prefer DataFrames for data engineering work - Catalyst can see every column expression:

In [ ]:
from pyspark.sql import functions as F

paid_orders = (
    orders
    .where(F.col("status") == "paid")
    .select("order_id", "customer_id", "amount", "order_ts")
)
paid_orders.orderBy("order_id").show()
assert [r["order_id"] for r in paid_orders.orderBy("order_id").collect()] == [1, 4, 7, 10]

Use RDDs only for record-level logic the DataFrame API cannot express. The lambda below is
opaque to the optimizer and runs in Python workers:

In [ ]:
log_lines = ["GET /orders 200", "POST /orders 201", "GET /health 200"]
line_lengths = spark.sparkContext.parallelize(log_lines).map(len)

print(line_lengths.collect())
assert line_lengths.collect() == [15, 16, 15]

## How do you read data with explicit schemas in PySpark?

Explicit schemas make pipelines safer and faster: inference can change with a small sample,
a malformed record, or a new partition. The cell below lands a small JSON file - one row
delivers `amount` as `"N/A"` - and reads it with a declared schema and a corrupt-record column.

In [ ]:
import json

raw_dir = os.path.join(WORK_DIR, "raw", "orders")
os.makedirs(raw_dir, exist_ok=True)
raw_rows = [
    {"order_id": "o1", "customer_id": "c1", "order_ts": "2026-10-04T09:15:00", "status": "paid", "amount": 120.00},
    {"order_id": "o2", "customer_id": "c2", "order_ts": "2026-10-04T10:05:00", "status": "shipped", "amount": 84.50},
    {"order_id": "o3", "customer_id": "c2", "order_ts": "2026-10-04T11:40:00", "status": "paid", "amount": "N/A"},
]
with open(os.path.join(raw_dir, "part-0.json"), "w") as handle:
    handle.write("\n".join(json.dumps(row) for row in raw_rows))

order_schema = T.StructType([
    T.StructField("order_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("amount", T.DecimalType(12, 2), True),
    T.StructField("_corrupt_record", T.StringType(), True),
])

orders_raw = (
    spark.read
    .schema(order_schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(raw_dir)
    .cache()  # Spark refuses queries that touch only the corrupt-record column of an uncached read
)

good_orders = orders_raw.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
bad_orders = orders_raw.filter(F.col("_corrupt_record").isNotNull())
print("good:", good_orders.count(), "corrupt:", bad_orders.count())
bad_orders.select("_corrupt_record").show(truncate=False)
assert good_orders.count() == 2 and bad_orders.count() == 1

For CSV, be explicit about headers, delimiters, quotes, and escapes:

In [ ]:
csv_dir = os.path.join(WORK_DIR, "raw", "customers")
os.makedirs(csv_dir, exist_ok=True)
with open(os.path.join(csv_dir, "part-0.csv"), "w") as handle:
    handle.write(
        "customer_id,email,created_at\n"
        'c1,"ada@example.com",2026-01-03 09:00:00\n'
        'c2,"grace, hopper@example.org",2026-01-04 10:30:00\n'
    )

customers_csv = (
    spark.read
    .schema("customer_id string, email string, created_at timestamp")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(csv_dir)
)
customers_csv.show(truncate=False)
assert customers_csv.where("customer_id = 'c2'").first()["email"] == "grace, hopper@example.org"

How it works under the hood. Reading JSON without a schema launches an extra job that samples files to infer types, so the data is touched twice. An explicit `StructType` skips that pass and lets the reader decode straight into the declared types. `mode` controls corrupt records: `PERMISSIVE` keeps the raw text in a corrupt-record column, `DROPMALFORMED` drops rows silently, and `FAILFAST` raises.

Worked example. Schema inference on a 2 TB JSON dataset costs one extra scan before the real job runs. If the last of three files delivers `amount` as `"N/A"`, inference over a sample can settle on `string`, while the explicit `decimal(12,2)` schema keeps the contract and turns the bad row into a countable null or corrupt record.

Common mistake: a mismatched type under `PERMISSIVE` produces nulls rather than an error, so always count the corrupt-record column (or nulls) after a load.

## What are transformations and actions in PySpark?

Transformations build a logical plan. Actions trigger execution.

Transformations only build a plan:

In [ ]:
daily_revenue = (
    orders
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)
print(type(daily_revenue).__name__, "- nothing has executed yet")

An action triggers execution. Each of these runs the whole lineage again:

In [ ]:
print("days with revenue:", daily_revenue.count())
daily_revenue.write.mode("overwrite").saveAsTable("daily_revenue")

total = spark.table("daily_revenue").agg(F.sum("revenue")).first()[0]
print("revenue written:", total)
assert float(total) == 2273.90

Common transformations include `select`, `where`, `withColumn`, `join`, `groupBy`, `agg`, `distinct`, and `dropDuplicates`. Common actions include `count`, `collect`, `show`, `take`, `write`, and `foreachBatch`.

How it works under the hood. Transformations only add nodes to the logical plan; nothing is scheduled. An action triggers Catalyst optimization, physical planning, a DAG split into stages at each shuffle, and one task per partition. That is why the same DataFrame walked twice by two actions is computed twice unless you cached it.

Worked example. A `groupBy("order_date").agg(sum)` over 500 GB shows `HashAggregate(partial) -> Exchange -> HashAggregate(final)`. The partial aggregate runs inside each map task and shrinks the rows before the network, so the shuffle moves far less than 500 GB; that two-node pattern is the signature of a decomposable aggregate.

Common mistake: calling `count()` or `show()` between every step to "check progress". Each call re-runs the whole lineage from the source; cache the frame or read `explain("formatted")` instead.

## How does lazy evaluation work in PySpark?

Lazy evaluation means Spark waits until an action before executing the plan. This lets Spark optimize the whole plan instead of executing every line immediately.

To see pushdown, write `orders` as Parquet and read it back - `explain` then shows what reaches
the file reader. The helper captures the plan text so a cell can assert on it.

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()


orders_path = os.path.join(WORK_DIR, "orders_parquet")
orders.write.mode("overwrite").parquet(orders_path)

result = (
    spark.read.parquet(orders_path)
    .where("order_ts >= timestamp '2026-01-10'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)
plan = plan_of(result)
print(plan)
read_schema = next(line for line in plan.splitlines() if line.startswith("ReadSchema"))
assert "GreaterThan(amount,0.00)" in plan  # the filter reached the Parquet reader
assert "status" not in read_schema  # unused columns are never read

Spark can combine filters, prune unused columns, push predicates to Parquet/Delta/Iceberg readers, and choose join strategies before running tasks.

Real-world example: if you read a 2 TB orders table but select only `order_id` and `amount`, Spark can avoid reading unused columns from Parquet files. That optimization is possible because the plan is declarative.

How it works under the hood. `explain("formatted")` prints the analyzed and optimized physical plan without reading data. Catalyst applies rule-based passes (predicate pushdown, constant folding, column pruning, null propagation) and the planner chooses join and aggregate strategies from table statistics. With AQE on, the printed plan is provisional and can change at runtime.

Worked example. `.where(ts >= ...).select(a, b).where(a > 0)` collapses into one `FileScan parquet` node with `PushedFilters: [IsNotNull(a), GreaterThan(a,0)]` and `ReadSchema: a,b`. Two filters and a projection became a single scan; no extra stage was created.

Common mistake: trusting the initial plan as final. AQE plans print `isFinalPlan=false` until an action runs, and `explain("cost")` shows the row estimates the planner used.

## How do you use Spark SQL and temp views from PySpark?

PySpark and Spark SQL work together. DataFrames can be registered as temporary views, and SQL results come back as DataFrames.

In [ ]:
orders.createOrReplaceTempView("orders")

daily_sql = spark.sql("""
select
  to_date(order_ts) as order_date,
  count(*) as order_count,
  sum(amount) as revenue
from orders
where status in ('paid', 'shipped')
group by to_date(order_ts)
""")

# the SQL and the DataFrame versions are the same plan, so they return the same rows
sql_rows = sorted((r["order_date"], r["revenue"]) for r in daily_sql.collect())
df_rows = sorted((r["order_date"], r["revenue"]) for r in daily_revenue.collect())
assert sql_rows == df_rows
daily_sql.orderBy("order_date").show()

SQL can create tables too. On Unity Catalog you would write
`create or replace table main.gold.daily_revenue as ...`; the local session catalog stores a
Parquet table:

In [ ]:
spark.sql("drop table if exists gold_daily_revenue")
spark.sql("""
create table gold_daily_revenue using parquet as
select to_date(order_ts) as order_date, sum(amount) as revenue
from orders
where status in ('paid', 'shipped')
group by to_date(order_ts)
""")
assert spark.table("gold_daily_revenue").count() == daily_sql.count()
print(spark.catalog.listTables())

Interview point: PySpark DataFrame code and Spark SQL usually compile into the same execution engine. Choose the style that is clearer for the team and workload.

How it works under the hood. `createOrReplaceTempView` registers a name in the session catalog that points at the DataFrame's logical plan. `spark.sql(...)` parses SQL into the same kind of plan tree, so both paths meet before Catalyst optimization; a DataFrame and its SQL twin usually produce identical physical plans. A global temp view lives under `global_temp` and is visible across sessions in the same application.

Worked example. Register `orders` and run the DataFrame aggregate and the SQL aggregate side by side; both `explain()` outputs show the same `HashAggregate + Exchange` shape because they compile to the same plan.

Common mistake: temp views are session-scoped and disappear with the session, so a job that re-registers a stale plan under the same name can silently query old logic. Drop the view or use a fresh name when the underlying frame changes.

## How do you clean and standardize data in PySpark?

Use built-in column functions instead of Python row loops. Built-ins run inside Spark's optimized execution engine.

In [ ]:
raw_customers = spark.createDataFrame(
    [
        ("  Ada@Example.COM ", " us", "2026-01-03 09:00:00", "+1 (312) 555-0100"),
        ("grace@example.org", "gb ", "2026-01-04 10:30:00", "020 7946 0018"),
        ("not-an-email", "us", "2026-01-05 11:00:00", None),
    ],
    "email string, country_code string, created_at string, phone string",
)

clean_customers = (
    raw_customers
    .select(
        F.trim(F.lower("email")).alias("email"),
        F.upper(F.trim("country_code")).alias("country_code"),
        F.to_timestamp("created_at").alias("created_at"),
        F.regexp_replace("phone", r"[^0-9]", "").alias("phone_digits"),
    )
    .withColumn("email_domain", F.element_at(F.split("email", "@"), -1))
    .where(F.col("email").contains("@"))
)
clean_customers.show(truncate=False)

rows = {r["email"]: r for r in clean_customers.collect()}
assert set(rows) == {"ada@example.com", "grace@example.org"}
assert rows["ada@example.com"]["phone_digits"] == "13125550100"
assert rows["grace@example.org"]["country_code"] == "GB"

For reusable business logic, wrap DataFrame transformations in plain Python functions:

In [ ]:
def standardize_orders(df):
    return (
        df
        .withColumn("status", F.lower(F.trim("status")))
        .withColumn("order_date", F.to_date("order_ts"))
        .where(F.col("order_id").isNotNull())
    )


standardized = standardize_orders(orders)
assert standardized.count() == orders.count() and "order_date" in standardized.columns

How it works under the hood. Built-in string and timestamp functions are Catalyst expressions evaluated on Tungsten rows inside the JVM; `regexp_replace` becomes a Java regex call per row with no Python involved. Wrapping transformations in plain Python functions is fine because those functions run once on the driver to build the plan, not once per row.

Worked example:

```text
"  Ada@Example.COM "  -> lower(trim(...))        -> "ada@example.com"
"+1 (312) 555-0100"   -> regexp_replace([^0-9])   -> "13125550100"
```

Common mistake: reaching for `F.udf` to trim, lowercase or split. The built-in version is shorter, faster, and keeps whole-stage code generation intact.

## Part 3 - Shape the data

You can now predict what will and will not execute. Part 3 is where frames meet data:
joins, aggregations and window functions are the wide operations that move bytes across
the cluster, and the cookbook closes it as the everyday recipe list. Every lesson below
reads the sample-data tables from Part 2, so the outputs shown are the outputs a reader
runs.

## How do you join data in PySpark?

A join combines DataFrames by key. The most important production questions are key uniqueness, null handling, data skew, and whether Spark should shuffle both sides or broadcast the small side.

In [ ]:
from pyspark.sql import functions as F

enriched = (
    orders.alias("o")
    .join(customers.alias("c"), on="customer_id", how="left")
    .select(
        "o.order_id",
        "o.customer_id",
        "c.name",
        "c.region",
        "o.status",
        "o.amount",
    )
    .orderBy("o.order_id")
)
enriched.show()

# `customer_id` is unique on the dimension, so the enrich join must not fan out
assert enriched.count() == orders.count()
print("join verified: one row per order")

If the dimension is small enough, broadcast it:

In [ ]:
from pyspark.sql.functions import broadcast

# eight tiny rows on the build side: every executor should hold `customers`,
# and nothing about `orders` needs to move at all
enriched = orders.join(broadcast(customers), "customer_id", "left")

assert enriched.count() == orders.count()

Real-world example: broadcasting a 20 MB customer segment dimension can avoid a large shuffle when joining it to a 3 TB orders fact table. Do not broadcast a dimension just because it is called a dimension; verify its size.

How it works under the hood. A sort-merge join shuffles both sides by the key hash, sorts each partition, then merges. A broadcast hash join ships the small side to every executor, builds a hash table there, and streams the large side past it with no shuffle at all. Spark picks broadcast when the estimated side size is under `spark.sql.autoBroadcastJoinThreshold`, and a `broadcast()` hint overrides the estimate.

In [ ]:
from pyspark.sql.functions import broadcast

small_customers = customers.select("customer_id", "name", "region")

enriched = (
    orders.alias("o")
    .join(broadcast(small_customers).alias("c"), on="customer_id", how="left")
    .select("o.order_id", "o.amount", "o.status", "c.name", "c.region")
)
plan = plan_of(enriched)
print(plan)
assert "BroadcastHashJoin" in plan and "SortMergeJoin" not in plan

Worked example. A 3 TB orders fact joined to a 20 MB customer dimension: sort-merge shuffles about 3 TB, while broadcast sends 20 MB to each executor and shuffles nothing. Across 100 executors that is about 2 GB of network instead of terabytes.

Common mistake: broadcasting a "dimension" that is actually 2 GB. Every executor must hold the whole build side, so you get OOM or a silently ignored hint; confirm `BroadcastHashJoin` in the plan.

## How do you aggregate data in PySpark?

Aggregations summarize records by one or more grouping columns. They usually create a shuffle because rows with the same key must be brought together.

In [ ]:
from pyspark.sql import functions as F

sku_sales = (
    order_items
    .withColumn("line_amount", F.col("qty") * F.col("unit_price"))
    .groupBy("sku")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.sum("qty").alias("units"),
        F.sum("line_amount").alias("revenue"),
    )
    .orderBy("sku")
)
sku_sales.show(20)

# SKU-4005 is carried by orders 5 and 12: two orders, two units, 55.25 + 55.00
row = sku_sales.where("sku = 'SKU-4005'").first()
assert row["orders"] == 2 and row["units"] == 2
assert round(float(row["revenue"]), 2) == 110.25

For conditional metrics, use expressions:

In [ ]:
from pyspark.sql import functions as F

customer_metrics = (
    orders
    .groupBy("customer_id")
    .agg(
        F.sum("amount").alias("lifetime_value"),
        F.sum(F.when(F.col("status") == "returned", F.col("amount")).otherwise(0)).alias("returned_amount"),
        F.max("order_ts").alias("last_order_ts"),
    )
    .orderBy("customer_id")
)
customer_metrics.show()

# customer 1: 120.00 + 84.50 + the fully returned 30.00 order
row = customer_metrics.where("customer_id = 1").first()
assert row["lifetime_value"] == 234.50 and row["returned_amount"] == 30.00
print("customers 7 and 8 are missing from the result - they have no orders to group")

Interview point: aggregations often need tuning because they shuffle data. Check the Spark UI for skewed shuffle partitions, spilled memory, and long-running reduce tasks.

How it works under the hood. A hash aggregate first runs a partial aggregate inside each map task, exchanges the reduced rows by group key, then runs the final aggregate. That partial step is why `sum` and `count` shuffle so much less than `countDistinct`, which must ship the distinct values themselves because they cannot be combined from partial sets exactly.

Worked example. 1 billion rows at 100 bytes each is 100 GB; a partial aggregate over 50 million keys can cut that to roughly 200 million intermediate rows, so the shuffle carries about 20 GB instead of 100 GB.

Common mistake: `countDistinct` on a high-cardinality column inside a big `groupBy`. Use `approx_count_distinct` when a small error is acceptable, or deduplicate first and count rows.

## How do window functions work in PySpark?

Window functions calculate values across related rows without collapsing the result like `groupBy` does.

In [ ]:
from pyspark.sql import Window, functions as F

w = Window.partitionBy("customer_id").orderBy(F.col("order_ts").desc())

latest_order = (
    orders
    .withColumn("rn", F.row_number().over(w))
    .where(F.col("rn") == 1)
    .drop("rn")
    .orderBy("customer_id")
)
latest_order.select("customer_id", "order_id", "status", "order_ts", "amount").show()

# the window kept one row per customer instead of collapsing the frame like groupBy
assert latest_order.count() == 6

Common window use cases:

+ Deduplicating CDC records by latest sequence number
+ Finding the latest row per business key
+ Calculating rolling metrics
+ Ranking products or customers within a segment

CDC example: several change rows per order, one replayed with the same LSN. Order by the
source sequence, then by ingestion time as a deterministic tie-break:

In [ ]:
from pyspark.sql import Window

orders_cdc = spark.createDataFrame(
    [
        (101, "placed",    1, datetime(2026, 1, 5, 9, 0)),
        (101, "paid",      2, datetime(2026, 1, 5, 9, 5)),
        (101, "shipped",   3, datetime(2026, 1, 6, 8, 0)),
        (102, "placed",    1, datetime(2026, 1, 5, 10, 0)),
        (102, "cancelled", 2, datetime(2026, 1, 5, 10, 1)),
        (102, "cancelled", 2, datetime(2026, 1, 5, 10, 9)),  # replayed change, later ingest
    ],
    "order_id int, status string, source_lsn long, ingested_at timestamp",
)

w = Window.partitionBy("order_id").orderBy(F.col("source_lsn").desc(), F.col("ingested_at").desc())

deduped_cdc = orders_cdc.withColumn("rn", F.row_number().over(w)).where("rn = 1").drop("rn")
deduped_cdc.orderBy("order_id").show()

latest = {r["order_id"]: (r["status"], r["ingested_at"].minute) for r in deduped_cdc.collect()}
assert latest == {101: ("shipped", 0), 102: ("cancelled", 9)}

How it works under the hood. A window with `partitionBy` plans as `Exchange hashpartitioning(key)` followed by `Sort` inside each partition, then the window expression runs in one pass. Window expressions without `partitionBy` send every row to a single partition, which turns a distributed job into one task. `row_number` needs the sort; `sum().over(rowsBetween(...))` reuses it.

In [ ]:
from pyspark.sql import Window, functions as F

seven_days = (
    Window.partitionBy("customer_id")
    .orderBy(F.col("order_ts").cast("long"))
    .rangeBetween(-7 * 86400, 0)
)

rolling = (
    orders
    .withColumn("spend_7d", F.sum("amount").over(seven_days))
    .orderBy("customer_id", "order_ts")
)
rolling.select("customer_id", "order_ts", "amount", "spend_7d").show()

# order 5 (placed, 55.25) sits inside the trailing week of order 4 (paid, 260.00)
row = rolling.where("order_id = 5").first()
assert float(row["spend_7d"]) == 315.25

Worked example. A 500 GB events table deduplicated with `row_number` over `partitionBy(order_id).orderBy(ts desc)` shuffles the whole table once, even if most keys are unique and there is nothing to deduplicate.

Common mistake: omitting a deterministic tie-break in `orderBy`, so the "latest" row changes between runs. Add the source sequence number or another column that breaks ties the same way every time.

## What are common PySpark data manipulation examples?

Most PySpark pipeline work is ordinary data manipulation: parsing timestamps, deriving dates, handling nulls, standardizing strings, splitting arrays, and creating business flags. Prefer `pyspark.sql.functions` because those expressions stay inside Spark's optimizer.

**Dates and timestamps.**

In [ ]:
from pyspark.sql import functions as F

orders_with_dates = (
    orders
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("order_month", F.date_trunc("month", "order_ts"))
    .withColumn("order_year", F.year("order_ts"))
    .withColumn("order_week", F.weekofyear("order_ts"))
    .withColumn("ship_by_date", F.date_add("order_date", 2))
    .withColumn("days_since_order", F.datediff(F.current_date(), "order_date"))
)
(
    orders_with_dates
    .orderBy("order_id")
    .select("order_id", "order_ts", "order_date", "ship_by_date", "days_since_order")
    .show()
)

# `ship_by_date` is `order_date` plus exactly two days, for every order
assert orders_with_dates.where(F.datediff("ship_by_date", "order_date") != 2).count() == 0

**Time zone conversion.** Common when sources emit UTC but reporting is local:

In [ ]:
from pyspark.sql import functions as F

localized = (
    orders
    .withColumn(
        "order_ts_chicago",
        F.from_utc_timestamp("order_ts", "America/Chicago"),
    )
    .withColumn("order_date_chicago", F.to_date("order_ts_chicago"))
    .orderBy("order_id")
)
localized.select("order_id", "order_ts", "order_ts_chicago", "order_date_chicago").show()

# order 12 lands at 00:30 UTC: in Chicago it is still January 13, in the evening
row = localized.where("order_id = 12").first()
assert row["order_ts_chicago"].strftime("%Y-%m-%d %H:%M") == "2026-01-13 18:30"

**Null handling and default values.**

In [ ]:
from pyspark.sql import functions as F

# customer 4's region never arrived; give the gap a default instead of a silent null
customers_filled = (
    customers
    .withColumn("region", F.coalesce("region", F.lit("unknown")))
)
customers_filled.orderBy("customer_id").select("customer_id", "name", "region").show()

row = customers_filled.where("customer_id = 4").first()
assert row["region"] == "unknown"
assert customers_filled.where(F.col("region").isNull()).count() == 0

**Type casting and safe derived columns.**

In [ ]:
typed_items = (
    order_items
    .withColumn("line_amount", (F.col("qty") * F.col("unit_price")).cast("decimal(12,2)"))
    .withColumn("unit_check", F.when(F.col("qty") > 0, F.col("line_amount") / F.col("qty")))
    .withColumn("bad_cast", F.lit("N/A").cast("int"))  # non-ANSI mode: an invalid cast is null
)
typed_items.select("order_id", "sku", "qty", "line_amount", "unit_check", "bad_cast").show(5)

assert typed_items.where(F.col("unit_check") != F.col("unit_price")).count() == 0
assert typed_items.where(F.col("bad_cast").isNotNull()).count() == 0

A cast that silently returns null is a data-quality bug waiting to happen. Spark 4 turns ANSI
mode on by default, which raises instead; with Spark 3.5 count the nulls after a cast.

**Conditional business flags.**

In [ ]:
from pyspark.sql import functions as F

classified = (
    orders
    .withColumn(
        "order_size",
        F.when(F.col("amount") >= 1000, "enterprise")
         .when(F.col("amount") >= 100, "mid_market")
         .otherwise("small"),
    )
    .withColumn(
        "is_completed",
        F.col("status").isin("paid", "shipped"),
    )
)
classified.orderBy("order_id").select("order_id", "amount", "status", "order_size", "is_completed").show()

# one wholesale order (1,180.00), five mid-market, six small
assert classified.where("order_size = 'enterprise'").count() == 1
assert classified.where("order_size = 'mid_market'").count() == 5

**Deduplicating records.**

In [ ]:
# the no-argument form compares entire rows, so exactly the retried copy dies
deduped_events = events.dropDuplicates()

assert events.count() == 13 and deduped_events.count() == 12
deduped_events.orderBy("user_id", "ts").show()

How it works under the hood. Every `withColumn` adds a projection node, and each `dropDuplicates` or window adds a wide node that exchanges rows by key. Catalyst later fuses the narrow projections into one, but a dedupe still moves the whole dataset across the network. `dropDuplicates(["order_id"])` repartitions by `order_id`; the no-argument form compares entire rows.

In [ ]:
from pyspark.sql import functions as F

order_summary = (
    orders
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("customer_id", "order_date")
    .agg(
        F.count("*").alias("orders"),
        F.sum("amount").alias("revenue"),
        F.max("amount").alias("largest_order"),
    )
    .withColumn("revenue_per_order", F.round(F.col("revenue") / F.col("orders"), 2))
    .orderBy("customer_id", "order_date")
)
order_summary.show()

# regrouping the ledger must not move money: bucketed revenue equals ledger revenue
assert order_summary.agg(F.sum("revenue")).first()[0] == orders.agg(F.sum("amount")).first()[0]

Worked example. Deduplicating a 1 TB orders table with `dropDuplicates(["order_id"])` shuffles roughly 1 TB by that key and keeps one arbitrary row per key.

Common mistake: expecting `dropDuplicates` to keep the newest record. It has no ordering, so the survivor is non-deterministic; for CDC use `row_number()` over a window ordered by a source sequence and a tie-break.

## How do you handle nested JSON and arrays in PySpark?

Nested data is common in event, clickstream, API, and CDC payloads. Use struct and array functions instead of converting rows to Python dictionaries.

In [ ]:
nested_dir = os.path.join(WORK_DIR, "raw", "events")
os.makedirs(nested_dir, exist_ok=True)
nested_rows = [
    {"event_id": "e1", "user": {"id": 1}, "device": {"os": "ios"},
     "items": [{"sku": "SKU-4001", "quantity": 2}, {"sku": "SKU-4007", "quantity": 1}],
     "payload_json": '{"source": "email", "campaign": {"id": "c-9", "name": "winter"}}'},
    {"event_id": "e2", "user": {"id": 2}, "device": {"os": "android"},
     "items": [{"sku": "SKU-4002", "quantity": 1}],
     "payload_json": '{"source": "ads", "campaign": {"id": "c-3", "name": "search"}}'},
    {"event_id": "e3", "user": {"id": 3}, "device": {"os": "web"}, "items": None,
     "payload_json": '{"source": "direct", "campaign": null}'},
]
with open(os.path.join(nested_dir, "part-0.json"), "w") as handle:
    handle.write("\n".join(json.dumps(row) for row in nested_rows))

raw_events = spark.read.schema(
    "event_id string, user struct<id:int>, device struct<os:string>, "
    "items array<struct<sku:string,quantity:int>>, payload_json string"
).json(nested_dir)

flattened = raw_events.select(
    "event_id",
    F.col("user.id").alias("user_id"),
    F.col("device.os").alias("device_os"),
    F.explode_outer("items").alias("item"),
)
items = flattened.select(
    "event_id", "user_id", F.col("item.sku").alias("sku"), F.col("item.quantity").alias("quantity")
)
items.orderBy("event_id", "sku").show()

# explode drops e3 (null array); explode_outer keeps it with null item columns
assert raw_events.select(F.explode("items")).count() == 3
assert items.count() == 4

If a payload column contains JSON strings, parse it with `from_json`:

In [ ]:
payload_schema = "source string, campaign struct<id:string,name:string>"

parsed = (
    raw_events
    .withColumn("payload", F.from_json("payload_json", payload_schema))
    .select("event_id", F.col("payload.source").alias("source"), F.col("payload.campaign.id").alias("campaign_id"))
)
parsed.orderBy("event_id").show()
assert {r["event_id"]: r["campaign_id"] for r in parsed.collect()} == {"e1": "c-9", "e2": "c-3", "e3": None}

Interview point: `explode` can multiply rows. Always know the expected array size and watch for unexpected row explosions.

How it works under the hood. Parquet and JSON readers expose nested objects as `struct` and `array` columns, and `F.col("user.id")` resolves the field at plan time with no data conversion. `explode` is a `Generate` operator that expands one row into many inside its partition; it is narrow, but it multiplies the row count, so every later shuffle and write processes more data.

Worked example. An events file with an average array length of 4 and 100 million events becomes 400 million rows after `explode`, so a downstream join or write handles four times the rows and roughly four times the bytes.

Common mistake: choosing `explode` when `explode_outer` is correct. Rows with a null or empty array simply disappear, which is a classic silent correctness bug in event pipelines.

## Part 4 - Production patterns

The recipes above assume someone handed you clean tables. A production job asks who
wrote them and how: governed writes, incremental loads, tests, bad records, streaming,
monitoring, and the tuning levers that keep a nightly job boring.

## How do you write PySpark tables?

With Unity Catalog, prefer table names like `catalog.schema.table` over raw paths in
application code, so governance, lineage, and permissions live at the table layer. The local
catalog has two levels (`schema.table`) but the habit is the same:

In [ ]:
daily_product_sales = (
    order_items
    .join(orders.select("order_id", "order_ts"), "order_id")
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date", "sku")
    .agg(F.sum("qty").alias("units"), F.sum(F.col("qty") * F.col("unit_price")).alias("revenue"))
)

spark.sql("create database if not exists gold")
daily_product_sales.write.mode("overwrite").saveAsTable("gold.daily_product_sales")
assert spark.table("gold.daily_product_sales").count() == daily_product_sales.count()

For partitioned tables add `partitionBy` (on Databricks add `.format("delta")`), and append
to an existing table with `mode("append")`. `df.writeTo("catalog.schema.table").append()` is the
v2 API for Delta and Iceberg catalogs; the local Parquet catalog only supports the v1 writer.

In [ ]:
(
    daily_product_sales
    .write
    .mode("overwrite")
    .partitionBy("order_date")
    .saveAsTable("gold.daily_product_sales_by_day")
)
partitions = spark.sql("show partitions gold.daily_product_sales_by_day").count()
print("partitions:", partitions)
assert partitions == daily_product_sales.select("order_date").distinct().count()

How it works under the hood. `saveAsTable` goes through the catalog, which owns the schema, permissions and lineage. `writeTo(...).append()` works only when the table already exists and uses the table's stored schema and partitioning. On a partitioned table, `mode("overwrite")` replaces the whole table unless `spark.sql.sources.partitionOverwriteMode` is `dynamic`, in which case only partitions present in the DataFrame are replaced.

Dynamic overwrite in practice: a pricing bug on 12 January is fixed, and only that day is
rewritten - the other days keep their files:

In [ ]:
sales_path = os.path.join(WORK_DIR, "daily_product_sales")
daily_product_sales.write.mode("overwrite").partitionBy("order_date").parquet(sales_path)
other_days_before = spark.read.parquet(sales_path).where("order_date != '2026-01-12'").agg(F.sum("revenue")).first()[0]

corrected = (
    daily_product_sales
    .where("order_date = '2026-01-12'")
    .withColumn("revenue", F.col("revenue") * F.lit(0.9))  # the reprocessed day
)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
corrected.write.mode("overwrite").partitionBy("order_date").parquet(sales_path)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

after = spark.read.parquet(sales_path)
other_days_after = after.where("order_date != '2026-01-12'").agg(F.sum("revenue")).first()[0]
assert after.select("order_date").distinct().count() == daily_product_sales.select("order_date").distinct().count()
assert other_days_after == other_days_before
print("only 2026-01-12 was rewritten; the other days are untouched")

Worked example. A table partitioned by `order_date` with 365 days and one new day of data: a static overwrite rewrites all 365 partitions, a dynamic overwrite touches only the one partition present in the frame.

Common mistake: leaving `overwriteSchema` on while an upstream type drifts. The write succeeds and quietly changes the table contract; own the schema in the pipeline and fail fast instead.

## How do you build an incremental PySpark pipeline?

An incremental pipeline processes only new or changed records. The usual pattern is to use a watermark, ingestion timestamp, source sequence number, or table change feed.

A batch watermark lives in an ops table. Read it, take only newer bronze rows, keep the latest
change per key, then upsert. Delta and Iceberg provide `MERGE`:

```sql
merge into main.silver.orders as target
using staged_orders as source
on target.order_id = source.order_id
when matched then update set *
when not matched then insert *
```

Plain Parquet has no `MERGE`, so the cell below expresses the same upsert as "rows whose key did
not change, plus the staged rows" - and runs it twice to prove it is idempotent.

In [ ]:
watermarks = spark.createDataFrame(
    [("orders_silver", datetime(2026, 1, 10, 0, 0))],
    "pipeline_name string, loaded_until_ts timestamp",
)
last_loaded_ts = (
    watermarks.where(F.col("pipeline_name") == "orders_silver")
    .select(F.max("loaded_until_ts").alias("ts"))
    .first()["ts"]
)

silver = orders.where(F.col("order_ts") <= F.lit(last_loaded_ts))  # what the last run loaded

bronze_changes = (
    orders.where(F.col("order_ts") > F.lit(last_loaded_ts))
    .withColumn("source_lsn", F.lit(1))
    .withColumn("ingested_at", F.col("order_ts"))
    .unionByName(spark.createDataFrame(  # order 8 was refunded, and the change arrived twice
        [(8, 4, "refunded", datetime(2026, 1, 10, 19, 30), D("89.90"), 2, datetime(2026, 1, 15, 8, 0)),
         (8, 4, "refunded", datetime(2026, 1, 10, 19, 30), D("89.90"), 2, datetime(2026, 1, 15, 8, 5))],
        T.StructType(silver.schema.fields + [T.StructField("source_lsn", T.IntegerType()), T.StructField("ingested_at", T.TimestampType())]),
    ))
)

staging_window = Window.partitionBy("order_id").orderBy(F.col("source_lsn").desc(), F.col("ingested_at").desc())
staged = (
    bronze_changes.withColumn("rn", F.row_number().over(staging_window))
    .where("rn = 1")
    .drop("rn", "source_lsn", "ingested_at")
)


def upsert(target, changes, key="order_id"):
    untouched = target.join(changes.select(key), key, "left_anti")
    return untouched.unionByName(changes)


once = upsert(silver, staged)
twice = upsert(once, staged)

assert once.count() == orders.count()
assert once.exceptAll(twice).count() == 0 and twice.exceptAll(once).count() == 0
assert once.where("order_id = 8").first()["status"] == "refunded"
print("rerunning the same batch leaves silver unchanged")

Interview point: a reliable incremental job needs idempotency. If the job retries the same batch, the target should end in the same state, not duplicate data.

How it works under the hood. The watermark read (`max(loaded_until_ts)`) and the `MERGE` are separate actions: the first is a small job that returns one row to the driver, the second is a full job. Delta and Iceberg implement `MERGE` by rewriting the files that contain matched keys and committing a new table version, so idempotency comes from matching on the business key, not from row-by-row updates.

Worked example. A batch of 5 million changed orders merging into a 2 billion row table: with partition pruning on `order_date`, `MERGE` rewrites only the files in the touched partitions rather than the whole table. The job takes minutes instead of hours.

Common mistake: deriving the watermark from `max(ingested_at)` on the target itself. Late data that arrives after that value is skipped forever; keep the watermark in an ops table and use a deliberate lookback window.

## How do you test PySpark transformations?

Keep business transformations as functions that accept and return DataFrames. Then create small test DataFrames and compare outputs.

In [ ]:
def add_order_date(df):
    return df.withColumn("order_date", F.to_date("order_ts"))


def test_add_order_date():
    input_df = spark.createDataFrame([("o1", datetime(2026, 10, 4, 12, 30, 0))], ["order_id", "order_ts"])
    actual = add_order_date(input_df).select("order_id", "order_date").collect()
    assert str(actual[0]["order_date"]) == "2026-10-04"


def test_add_order_date_keeps_nulls():
    input_df = spark.createDataFrame(
        [("o1", "2026-10-04 12:30:00"), ("o2", None)],
        "order_id string, order_ts string",
    )
    rows = add_order_date(input_df).orderBy("order_id").collect()
    assert [r["order_id"] for r in rows] == ["o1", "o2"]
    assert str(rows[0]["order_date"]) == "2026-10-04"
    assert rows[1]["order_date"] is None


for test in (test_add_order_date, test_add_order_date_keeps_nulls):
    test()
    print("passed:", test.__name__)

For real projects, use `pytest`, fixtures for the Spark session, and small deterministic inputs. Test data should cover nulls, duplicate keys, late-arriving data, invalid records, and boundary timestamps.

How it works under the hood. A DataFrame test needs a session, but `createDataFrame` from local rows builds a small deterministic frame with no cluster involved. The test should compare collected rows in a defined order and also assert on schema and nullability, because a transformation that changes types is as much a bug as one that changes values.

Worked example. The one-row test above builds a one-partition local frame, runs a single action and finishes in milliseconds, so a suite of them fits in a normal CI run against one shared local `SparkSession` fixture.

Common mistake: asserting on `collect()` order without an `orderBy`. Spark does not guarantee row order, so the test passes locally and fails in CI for no real reason.

## How do you handle bad records and data quality checks in PySpark?

Bad records should be isolated, measurable, and visible. Do not silently drop them unless the product owner agrees that loss is acceptable.

In [ ]:
bronze_orders_raw = spark.createDataFrame(
    [(1, 10, D("50.00")), (2, None, D("20.00")), (None, 11, D("5.00")), (4, 12, D("-3.00")), (5, 13, D("8.00"))],
    "order_id int, customer_id int, amount decimal(12,2)",
)

checked = bronze_orders_raw.withColumn(
    "quality_error",
    F.when(F.col("order_id").isNull(), "missing_order_id")
     .when(F.col("customer_id").isNull(), "missing_customer_id")
     .when(F.col("amount") < 0, "negative_amount"),
).cache()  # both writes below read the same checked frame

valid_orders = checked.where("quality_error is null").drop("quality_error")
quarantine = checked.where("quality_error is not null").withColumn("rejected_at", F.current_timestamp())

valid_orders.write.mode("overwrite").saveAsTable("silver_orders_checked")
quarantine.write.mode("overwrite").saveAsTable("ops_orders_quarantine")

quarantine.groupBy("quality_error").count().orderBy("quality_error").show()
assert valid_orders.count() == 2 and quarantine.count() == 3
checked.unpersist()

Interview point: quality checks should happen before publishing gold tables. A dashboard should not discover broken data before the pipeline does.

How it works under the hood. A rule column evaluated once gives one plan with two filters over the same scan, so both the valid and the quarantine write read the source. `subtract` (and `exceptAll`) is a wide operation that compares whole rows and shuffles both sides, so it is heavier than an explicit `is_valid` flag and behaves differently once you add a timestamp column.

Worked example. 10 million raw rows with 200 thousand bad ones: caching the checked frame and writing valid plus quarantine reads the source once at the cost of holding the frame; recomputing reads twice but needs no cache memory.

Common mistake: using `subtract` after adding a `rejected_at` timestamp. The columns no longer match, so no row cancels out and the quarantine gets everything. Derive a boolean and filter it twice.

## How do you use UDFs safely in PySpark?

Use Spark built-in functions first. Python UDFs often move data between the JVM and Python worker, which can be slower and harder for Spark to optimize.

Prefer the built-in expression, and compare it with the Python UDF that does the same job:

In [ ]:
from pyspark.sql import types as T


@F.udf(T.StringType())
def email_domain_udf(email):
    if email is None or "@" not in email:
        return None
    return email.strip().lower().split("@")[-1]


builtin = raw_customers.withColumn(
    "email_domain",
    F.when(F.col("email").contains("@"), F.element_at(F.split(F.lower(F.trim("email")), "@"), -1)),
)
with_udf = raw_customers.withColumn("email_domain", email_domain_udf("email"))

assert sorted(map(tuple, builtin.collect())) == sorted(map(tuple, with_udf.collect()))
udf_plan = plan_of(with_udf)
assert "BatchEvalPython" in udf_plan and "BatchEvalPython" not in plan_of(builtin)
print("same answer; only the UDF plan crosses into Python (BatchEvalPython)")

If you need Python for vectorized numeric or ML work, a pandas UDF moves Arrow batches instead
of single rows (it needs `pandas` and `pyarrow` on every worker), but measure it:

```python
import pandas as pd
from pyspark.sql import functions as F, types as T

@F.pandas_udf(T.DoubleType())
def cap_score(score: pd.Series) -> pd.Series:
    return score.clip(lower=0.0, upper=1.0)

scored = users.withColumn("capped_score", cap_score("score"))
```

How it works under the hood. A Python UDF plans as a `BatchEvalPython` node: each task starts a Python worker, sends batches of rows over a socket, and receives results, usually via Arrow or pickles. Catalyst cannot push predicates through that boundary and whole-stage code generation stops there, so the UDF is a black box in the middle of an otherwise compiled pipeline.

Worked example. For 100 million rows, a row-at-a-time UDF costing one microsecond of Python per row adds about 100 seconds of interpreter time before you count serialization, and it blocks the generated code path around it.

Common mistake: an `@F.udf` closure that captures a large lookup dict or DataFrame. The object is pickled to every task; broadcast it as a Spark broadcast variable instead, or express the logic with built-ins.

## How do you process streaming data with PySpark Structured Streaming?

Structured Streaming lets you express streaming logic with DataFrame APIs. Spark runs the query continuously or in micro-batches.

The cell below lands two JSON files in a directory (the second repeats an event), streams them
with a watermark and `dropDuplicatesWithinWatermark`, and upserts each micro-batch into a
key-value sink with `foreachBatch`. `trigger(availableNow=True)` processes everything available
and stops - the scheduled-batch pattern. On Databricks the source would be Auto Loader
(`format("cloudFiles")`) and the sink a Delta `MERGE`.

In [ ]:
landing_dir = os.path.join(WORK_DIR, "landing", "events")
checkpoint_dir = os.path.join(WORK_DIR, "checkpoints", "events_silver")
os.makedirs(landing_dir, exist_ok=True)
landed = [
    [("e1", "u1", "2026-01-06T09:00:00", "view"), ("e2", "u1", "2026-01-06T09:02:00", "add_to_cart")],
    [("e2", "u1", "2026-01-06T09:02:00", "add_to_cart"), ("e3", "u2", "2026-01-06T09:05:00", "purchase")],
]
columns = ["event_id", "user_id", "event_ts", "event_type"]
for index, batch in enumerate(landed):
    with open(os.path.join(landing_dir, f"part-{index}.json"), "w") as handle:
        handle.write("\n".join(json.dumps(dict(zip(columns, row))) for row in batch))

silver_events = {}  # stands in for a table keyed by event_id
batch_ids = []


def merge_batch(batch_df, batch_id):
    batch_ids.append(batch_id)
    silver_events.update({row["event_id"]: row.asDict() for row in batch_df.collect()})


def run_stream():
    query = (
        spark.readStream
        .schema("event_id string, user_id string, event_ts timestamp, event_type string")
        .json(landing_dir)
        .withWatermark("event_ts", "30 minutes")
        .dropDuplicatesWithinWatermark(["event_id"])
        .writeStream
        .foreachBatch(merge_batch)
        .option("checkpointLocation", checkpoint_dir)
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()


run_stream()
print("silver events:", sorted(silver_events), "batches:", batch_ids)
assert sorted(silver_events) == ["e1", "e2", "e3"]

batches_before = len(batch_ids)
run_stream()  # same checkpoint: the files are already committed, so nothing is reprocessed
assert len(batch_ids) == batches_before
print("second run read nothing new - the checkpoint remembers progress")

Real-world points:

+ Checkpoint locations are required for fault tolerance.
+ Watermarks define how late data can arrive before Spark drops state.
+ Deduplication needs a stable event key.
+ Streaming writes must be idempotent or carefully checkpointed.

How it works under the hood. A streaming query is a batch plan re-executed on every trigger. State for deduplication and aggregations lives in the state store under `checkpointLocation`, and a watermark bounds how long that state is kept. The driver keeps the query running and schedules one micro-batch job per trigger; the sink commits each batch transactionally.

Worked example. A 5-minute tumbling window with a 30-minute watermark keeps at most about six windows of state per key; drop the watermark and the state store grows for the life of the query.

Common mistake: `outputMode("complete")` with an unbounded key space, which recomputes and rewrites the whole result on every batch. Use `append` or `update` with a watermark for streaming sinks.

## How do you monitor and debug PySpark jobs?

Start with the Spark UI. It shows jobs, stages, tasks, shuffle read/write, spilled memory, input size, executor failures, and SQL plans.

Useful PySpark checks:

In [ ]:
print(plan_of(enriched, "simple"))
print("partitions:", enriched.rdd.getNumPartitions())
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Common symptoms:

+ One task runs much longer than others: likely skew
+ Huge shuffle read/write: expensive join, aggregation, or repartition
+ Spilled memory: insufficient executor memory or too-wide shuffle
+ Many tiny tasks: small files or too many partitions
+ Driver out of memory: collecting too much data to the driver

For production jobs, log record counts at each pipeline boundary:

In [ ]:
input_count = bronze_orders_raw.count()
output_count = valid_orders.count()

spark.createDataFrame(
    [("orders_silver", input_count, output_count)],
    "pipeline_name string, input_count long, output_count long",
).write.mode("append").saveAsTable("ops_pipeline_metrics")

spark.table("ops_pipeline_metrics").show()
assert (input_count, output_count) == (5, 2)

How it works under the hood. The driver's DAG scheduler tracks stages and tasks and emits the event log the Spark UI reads. The numbers that matter are per-stage: shuffle read and write bytes, spill to memory and disk, the task duration distribution, and GC time. A stage boundary exists wherever the plan contains an `Exchange`.

Worked example. If 199 tasks in a stage finish in 20 seconds and one takes 600 seconds, the stage wall time is 600 seconds. Adding partitions will not help; the hot key must be split with AQE skew join or manual salting.

Common mistake: looking only at the average task time. The max-to-median ratio is the skew signal, and excessive driver-side `collect()` shows up as driver GC, not as a slow task.

## How do you make a PySpark job faster?

Tuning starts with the plan and the Spark UI, not with a config change. Find the widest stage, read its shuffle bytes, spill and task-duration spread, then fix the one thing that dominates. The levers below are ordered by how often they matter. The deeper recipes live in [Spark Tuning](spark-tuning.ipynb).

**Adaptive Query Execution (AQE).** AQE re-optimizes the plan after shuffle stages report real sizes. It coalesces many small post-shuffle partitions, splits oversized partitions for skewed joins, and can switch a sort-merge join to a broadcast join at runtime. It is on by default since Spark 3.2.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
print({key: spark.conf.get(key) for key in ("spark.sql.adaptive.enabled", "spark.sql.adaptive.skewJoin.enabled")})

**Shuffle partition sizing.** `spark.sql.shuffle.partitions` sets the reduce fan-out for every shuffle; the default 200 is wrong for both tiny and huge jobs. Aim for partitions of roughly 128-256 MB after the shuffle, and let AQE coalesce the small ones instead of hand-tuning per query.

In [ ]:
shuffle_bytes = 200 * 1024 ** 3
target_partition_bytes = 128 * 1024 ** 2
partitions = -(-shuffle_bytes // target_partition_bytes)  # ceiling division
print("target partitions:", partitions)

**Broadcast versus sort-merge join.** A broadcast join ships the small side to every executor and avoids shuffling the large side; it is chosen when the estimated side is under `spark.sql.autoBroadcastJoinThreshold` (10 MB by default). Fall back to sort-merge when neither side is small, and check the physical plan to confirm what was chosen.

In [ ]:
threshold_bytes = int(spark.conf.get("spark.sql.autoBroadcastJoinThreshold").rstrip("b"))
print("broadcast threshold bytes:", threshold_bytes)

enriched = orders.join(broadcast(customers), on="customer_id", how="left")
assert "BroadcastHashJoin" in plan_of(enriched)
assert threshold_bytes == 10 * 1024 * 1024

**Skew.** One hot key puts most rows in one partition, so one task defines the stage duration. AQE skew join splits it when the partition exceeds both `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (256 MB) and 5x the median size. Otherwise, salt the hot key: add a random bucket to the fact side and explode the dimension across the same buckets.

In [ ]:
salt_buckets = 16
salted_facts = orders.withColumn("salt", (F.rand(seed=42) * salt_buckets).cast("int"))
salted_dim = customers.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(salt_buckets)])))
joined = salted_facts.join(salted_dim, ["customer_id", "salt"]).drop("salt")

# salting changes how rows are spread, never which rows match
assert joined.count() == orders.join(customers, "customer_id").count()
print("salted join rows:", joined.count())

**Caching.** Cache only a frame that several actions reuse or that feeds an iterative loop. Caching is lazy: the first action materialises the blocks, and you should `unpersist()` when done. A cached frame used once just adds memory pressure.

In [ ]:
enriched = orders.join(broadcast(customers), "customer_id").cache()
enriched.count()  # the first action materializes the cached blocks
enriched.groupBy("region").count().orderBy("region").show()
assert enriched.is_cached
enriched.unpersist()

**File layout and predicate pushdown.** Partition on a low-cardinality column that appears in filters, target 100 MB-1 GB per file, and write a sensible number of files with `repartition` or `coalesce`. On the read side, `spark.sql.files.maxPartitionBytes` (128 MB) controls how much one task reads, and Parquet or Delta statistics skip files and row groups that cannot match the predicate.

In [ ]:
curated_path = os.path.join(WORK_DIR, "curated_orders")
(
    orders
    .withColumn("order_date", F.to_date("order_ts"))
    .repartition("order_date")
    .write.mode("overwrite")
    .partitionBy("order_date")
    .parquet(curated_path)
)

pruned = spark.read.parquet(curated_path).filter("order_date = '2026-01-12'")
plan = plan_of(pruned)
print(plan)
assert "PartitionFilters: [isnotnull(order_date" in plan
assert sorted(r["order_id"] for r in pruned.collect()) == [9, 10]

That is the short list. For the full treatment of AQE, partition sizing, join strategies, skew, storage layout and profiling, see [Spark Tuning](spark-tuning.ipynb).

## Part 5 - Real world and interview

Five use cases show where the patterns above pay off - nightly ELT, streaming ingestion,
feature tables, messy logs, safe backfills - then a section on how to say all of it in
an interview.

## Real world use cases for PySpark

### 1. Nightly batch ELT into a partitioned lake

Raw JSON lands in object storage, a Spark job cleans and types it, and the curated layer is
written as partitioned Parquet for downstream engines:

```python
orders = spark.read.schema(order_schema).json("s3://acme-raw/orders/ds=2026-10-04/")
(
    orders.where("amount_cents > 0")
    .withColumn("amount", F.col("amount_cents") / 100)
    .repartition("order_date")            # one file family per day
    .sortWithinPartitions("country")      # tight min/max for skipping
    .write.mode("overwrite")
    .option("compression", "zstd")
    .partitionBy("order_date")
    .parquet("s3://acme-curated/orders/")
)
```

Why Spark: a few minutes of distributed CPU replaces hours of single-node scripting, and explicit schemas remove inference surprises. Enable Adaptive Query Execution (`spark.sql.adaptive.enabled=true`) so skewed joins and partition counts self-tune.

### 2. Structured Streaming from Kafka into a lakehouse table

A streaming job decodes Kafka events, applies the batch transformations, and merges
micro-batches into an Iceberg or Delta table:

```python
(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "earliest")
    .load()
    .select(F.from_json(F.col("value").cast("string"), order_schema).alias("o"))
    .select("o.*")
    .writeStream
    .foreachBatch(merge_into_lakehouse)          # idempotent MERGE per batch
    .option("checkpointLocation", "s3://acme-checkpoints/orders/")
    .trigger(processingTime="1 minute")
    .start()
)
```

Why Spark: the batch and streaming APIs share transformations, so there is one business-logic implementation. Checkpointing plus an idempotent merge gives end-to-end exactly-once.

### 3. Feature engineering for machine learning

Feature tables use broadcast joins to small dimensions and vectorized pandas UDFs instead of
row-at-a-time Python UDFs:

```python
@pandas_udf("double")
def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

features = (
    events.join(F.broadcast(products), "product_id")
    .groupBy("customer_id")
    .agg(F.count("*").alias("orders_30d"), F.sum("amount").alias("spend_30d"))
    .withColumn("spend_z", zscore("spend_30d"))
)
```

Why Spark: the same feature definitions can be applied to historical training data and to daily scoring runs, which reduces train/serve skew.

### 4. Log and event processing with messy input

Semi-structured logs break schema assumptions. Read permissively (as in the corrupt-record
cell above), write good rows to the curated table and bad rows to a quarantine path, and evolve
the schema deliberately.

Why Spark: quarantine plus schema-on-read lets ingestion continue while the platform team fixes the producer, instead of failing the whole day.

### 5. Backfills and reprocessing without duplicating data

A bug fix requires reprocessing September. Set
`spark.sql.sources.partitionOverwriteMode=dynamic` and overwrite with only September's rows: the
dynamic-overwrite cell in Part 4 shows exactly this on one day.

Why Spark: the same job that handles daily loads can reprocess a month, and dynamic overwrite makes the rerun idempotent at partition granularity - the property that makes backfills safe.

## What should you say in a PySpark interview answer?

A strong PySpark interview answer should include the API, the execution model, and the production concern.

Example answer for "How do you deduplicate orders?": `row_number()` over
`Window.partitionBy("order_id").orderBy(F.col("source_lsn").desc())`, keep `rn = 1` (the CDC
cell above), then explain:

+ `row_number` keeps one row per business key.
+ Ordering must use source metadata, not random ingestion order.
+ The result should be merged into the target for idempotency.
+ If one key is extremely hot, this window can skew and needs tuning.

How it works under the hood, restated as an answer structure. A strong answer names the API, the execution model, and the failure mode. For deduplication: "I use `row_number()` over `Window.partitionBy(business_key).orderBy(source_lsn desc)` filtered to `rn = 1`, then `MERGE`; that forces one shuffle by key, and if one key is hot the window skews, so I check the partition sizes and salt or let AQE skew join split it."

Worked example, a 30-second answer skeleton:

```text
1. API       - the exact functions and window spec you call
2. Engine    - narrow vs wide, where the shuffle is, partition count
3. Ops       - idempotency, skew, cost, monitoring, failure recovery
```

Common mistake: answering with syntax only. Interviewers probe the plan, so tie every method to a shuffle, a partition count, or a failure mode.

## Practice

The guided practice notebook works through the same ideas cell by cell with assert checks
after each one.

## Practice: PySpark concepts

The hands-on practice for this topic lives in its own notebook: [`04_pyspark_concepts.ipynb`](../04_pyspark_concepts.ipynb) — [Open in Colab](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/04_pyspark_concepts.ipynb).